# Persistence Diagrams: Visualisation, Distances, and Vectorisation

Persistence diagrams are the primary output of persistent homology. This notebook covers:
1. **Visualisation** -- diagrams, barcodes, landscapes
2. **Distances** -- bottleneck and Wasserstein distances
3. **Vectorisation** -- persistence images and landscapes for ML pipelines

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from ripser import ripser
    from persim import plot_diagrams, bottleneck, wasserstein, PersistenceImager
    HAS_TDA = True
except ImportError:
    HAS_TDA = False
    print('Install: pip install ripser persim')

%matplotlib inline
plt.rcParams['figure.figsize'] = (7, 5)

In [ ]:
# Generate three point clouds with different topology
np.random.seed(42)
n = 150

# Circle
t = 2 * np.pi * np.random.rand(n)
circle = np.column_stack([np.cos(t), np.sin(t)]) + 0.05 * np.random.randn(n, 2)

# Slightly perturbed circle (similar topology)
circle2 = np.column_stack([np.cos(t), np.sin(t)]) + 0.10 * np.random.randn(n, 2)

# Cluster (no loop)
cluster = 0.3 * np.random.randn(n, 2)

if HAS_TDA:
    dgm_c1 = ripser(circle, maxdim=1)['dgms']
    dgm_c2 = ripser(circle2, maxdim=1)['dgms']
    dgm_cl = ripser(cluster, maxdim=1)['dgms']
    print('Computed persistence for all three point clouds.')

## 1. Visualisation

A **persistence diagram** plots each feature as a point $(b, d)$ above the diagonal $b=d$. Distance from the diagonal equals the feature's lifetime.

In [ ]:
if HAS_TDA:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    titles = ['Circle (low noise)', 'Circle (high noise)', 'Gaussian cluster']
    for ax, dgm, title in zip(axes, [dgm_c1, dgm_c2, dgm_cl], titles):
        plot_diagrams(dgm, ax=ax, show=False)
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

## 2. Distances Between Diagrams

- **Bottleneck distance** $d_B$: the maximum cost of a matching between points.
- **Wasserstein distance** $W_p$: the $L^p$ cost of an optimal matching.

The **stability theorem** guarantees: small perturbations in the data produce small changes in the diagram.

In [ ]:
if HAS_TDA:
    # Compare H1 diagrams
    d_bn_similar = bottleneck(dgm_c1[1], dgm_c2[1])
    d_bn_diff = bottleneck(dgm_c1[1], dgm_cl[1])
    d_w_similar = wasserstein(dgm_c1[1], dgm_c2[1])
    d_w_diff = wasserstein(dgm_c1[1], dgm_cl[1])
    
    print(f"Bottleneck(circle1, circle2) = {d_bn_similar:.4f}  (similar topology)")
    print(f"Bottleneck(circle1, cluster) = {d_bn_diff:.4f}  (different topology)")
    print()
    print(f"Wasserstein(circle1, circle2) = {d_w_similar:.4f}")
    print(f"Wasserstein(circle1, cluster) = {d_w_diff:.4f}")
    print()
    print("As expected, similar shapes have small distance; different shapes have large distance.")

## 3. Vectorisation: Persistence Images

To use persistence diagrams in machine learning, we convert them to fixed-size vectors.

A **persistence image** places a weighted Gaussian at each point $(b, p)$ where $p = d - b$ is the persistence, then discretises on a grid.

In [ ]:
if HAS_TDA:
    pimgr = PersistenceImager(pixel_size=0.05, birth_range=(0, 0.5), pers_range=(0, 2.0))
    pimgr.fit([dgm_c1[1], dgm_c2[1], dgm_cl[1]])
    
    imgs = [pimgr.transform(d[1]) for d in [dgm_c1, dgm_c2, dgm_cl]]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    titles = ['Circle (low noise)', 'Circle (high noise)', 'Cluster']
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img, cmap='hot', origin='lower')
        ax.set_title(f'Persistence Image -- {title}')
    plt.tight_layout()
    plt.show()
    
    print(f"Image shape: {imgs[0].shape} -- ready for ML pipelines.")

In [ ]:
# Persistence landscape (manual computation)
if HAS_TDA:
    def persistence_landscape(dgm, t_values, k=1):
        """Compute the k-th persistence landscape function."""
        n_pts = len(dgm)
        landscapes = np.zeros((n_pts, len(t_values)))
        for i, (b, d) in enumerate(dgm):
            if np.isinf(d):
                continue
            mid = (b + d) / 2
            for j, t in enumerate(t_values):
                if b <= t <= mid:
                    landscapes[i, j] = t - b
                elif mid < t <= d:
                    landscapes[i, j] = d - t
        # k-th landscape: take the k-th largest value at each t
        landscapes.sort(axis=0)
        return landscapes[-k, :] if n_pts >= k else np.zeros(len(t_values))
    
    t_vals = np.linspace(0, 2.5, 200)
    fig, ax = plt.subplots()
    for dgm, label in [(dgm_c1[1], 'Circle'), (dgm_cl[1], 'Cluster')]:
        L1 = persistence_landscape(dgm, t_vals, k=1)
        ax.plot(t_vals, L1, label=label)
    ax.set_xlabel('t')
    ax.set_ylabel('Lambda_1(t)')
    ax.set_title('First Persistence Landscape (H1)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Key Takeaways

- **Persistence diagrams** are stable summaries of topological features.
- The **bottleneck** and **Wasserstein** distances quantify diagram similarity.
- **Persistence images** and **landscapes** convert diagrams into vectors for use in classifiers, clustering, etc.

**Next:** The Mapper algorithm for topological data visualisation.